Assignment for day 5

In [1]:
# imports
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# parameters
TICKERS = ["NVDA", "MSFT", "AAPL"]
START_DATE = "2025-01-01"
END_DATE = "2026-03-12"

# download OHLCV data for tickers
print(f"Downloading data for: {', '.join(TICKERS)}")
prices = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True)

/opt/miniconda3/envs/algotrading/lib/python3.11/site-packages/yfinance/scrapers/history.py:172: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
[*********************100%***********************]  3 of 3 completed


In [3]:
prices.head()

Price            Close                                High              \
Ticker            AAPL        MSFT        NVDA        AAPL        MSFT   
Date                                                                     
2025-01-02  242.525177  414.568604  138.264679  247.746654  421.986845   
2025-01-03  242.037827  419.292938  144.422684  242.853364  419.966414   
2025-01-06  243.668900  423.749786  149.381042  245.986242  430.157784   
2025-01-07  240.894073  418.322266  140.094086  244.215924  426.522914   
2025-01-08  241.381409  420.491241  140.064117  242.385931  422.878149   

Price                          Low                                Open  \
Ticker            NVDA        AAPL        MSFT        NVDA        AAPL   
Date                                                                     
2025-01-02  138.834500  240.506207  410.874369  134.585892  247.577564   
2025-01-03  144.852536  240.575812  415.519453  139.684231  242.037827   
2025-01-06  152.110159  241.878676  421.402504  147.771585  242.982646   
2025-01-07  153.079835  240.038745  416.767304  139.964123  241.659879   
2025-01-08  143.902856  238.745812  417.500194  137.514949  240.605648   

Price                                 Volume                       
Ticker            MSFT        NVDA      AAPL      MSFT       NVDA  
Date                                                               
2025-01-02  421.452012  135.955438  55740700  16896500  198247200  
2025-01-03  417.044673  139.964138  40244100  16662900  229322500  
2025-01-06  423.898343  148.541321  45045600  20573600  265377400  
2025-01-07  424.888733  152.979862  40856000  18139100  351782200  
2025-01-08  419.401777  142.533310  37628900  15054600  227349900

In [4]:
# save prices to csv files in market_data folder
df_prices = {}
for ticker in TICKERS:
    p_ticker = prices.loc[:, prices.columns.get_level_values("Ticker") == ticker]
    p_ticker.columns = p_ticker.columns.droplevel(1)
    p_ticker.to_csv(f"../../market_data/{ticker}.csv")
    df_prices[ticker] = p_ticker

In [9]:
for ticker in TICKERS:
    # sort index in ascending order
    df_prices[ticker] = df_prices[ticker].sort_index(ascending=True)
    # remove duplicates
    df_prices[ticker] = df_prices[ticker].drop_duplicates()

df_prices["NVDA"].head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2025-01-02,138.264679,138.834500,134.585892,135.955438,198247200
2025-01-03,144.422684,144.852536,139.684231,139.964138,229322500
2025-01-06,149.381042,152.110159,147.771585,148.541321,265377400
2025-01-07,140.094086,153.079835,139.964123,152.979862,351782200
2025-01-08,140.064117,143.902856,137.514949,142.533310,227349900


In [10]:
def backtest_sma_crossover(df_prices, fast: int = 12, slow: int = 26, signal=9, initial_capital: float = 10_000.0):
    """
    SMA crossover backtest. Enters long when fast MA crosses above slow MA,
    exits (goes flat) when fast crosses below. Trades execute at next day's open.

    Parameters
    ----------
    df_prices : pd.DataFrame
        DataFrame with at least 'Close' and 'Open' columns.
    fast : int
        Fast moving average window.
    slow : int
        Slow moving average window.
    initial_capital : float
        Starting cash in dollars.

    Returns
    -------
    pd.DataFrame with columns: signal, position, open, shares, cash, equity
    """
    df = df_prices.copy()
    df["fast_ma"] = df["Close"].ewm(span=fast, adjust=False).mean()
    df["slow_ma"] = df["Close"].ewm(span=slow, adjust=False).mean()

    # +1 when fast > slow, 0 otherwise (signal computed on close)
    df["signal"] = np.where(df["fast_ma"] > df["slow_ma"], 1, 0)

    # Shift signal by 1: trade at the NEXT day's open
    df["position"] = df["signal"].shift(1).fillna(0).astype(int)

    # Portfolio simulation
    cash = initial_capital
    shares = 0.0
    cash_list, shares_list, equity_list = [], [], []

    for _, row in df.iterrows():
        prev_pos = shares > 0

        # Enter
        if row["position"] == 1 and not prev_pos:
            shares = cash / row["Open"]
            cash = 0.0
        # Exit
        elif row["position"] == 0 and prev_pos:
            cash = shares * row["Open"]
            shares = 0.0

        cash_list.append(cash)
        shares_list.append(shares)
        equity_list.append(cash + shares * row["Close"])

    result = df.copy()
    result["cash"] = cash_list
    result["shares"] = shares_list
    result["equity"] = equity_list
    result["returns"] = result["equity"].pct_change()
    result["drawdown"] = result["equity"] / result["equity"].cummax() - 1
    return result


def print_stats(result, initial_capital=10_000.0):
    total_return = result["equity"].iloc[-1] / initial_capital - 1
    ann_return = (1 + total_return) ** (252 / len(result)) - 1
    ann_vol = result["returns"].std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol > 0 else np.nan
    max_dd = result["drawdown"].min()
    num_trades = result["position"].diff().abs().sum() / 2

    print(f"Total Return:    {total_return:.1%}")
    print(f"Ann. Return:     {ann_return:.1%}")
    print(f"Ann. Volatility: {ann_vol:.1%}")
    print(f"Sharpe Ratio:    {sharpe:.2f}")
    print(f"Max Drawdown:    {max_dd:.1%}")
    print(f"Num Trades:      {int(num_trades)}")


In [12]:
results = {}
for ticker in TICKERS:
    print(f"Backtesting {ticker}")
    results[ticker] = backtest_sma_crossover(df_prices[ticker])
    print_stats(results[ticker])

Backtesting NVDA
Total Return:    -1.7%
Ann. Return:     -1.5%
Ann. Volatility: 31.9%
Sharpe Ratio:    -0.05
Max Drawdown:    -30.6%
Num Trades:      9
Backtesting MSFT
Total Return:    19.9%
Ann. Return:     16.6%
Ann. Volatility: 14.9%
Sharpe Ratio:    1.11
Max Drawdown:    -9.5%
Num Trades:      3
Backtesting AAPL
Total Return:    -4.9%
Ann. Return:     -4.2%
Ann. Volatility: 16.9%
Sharpe Ratio:    -0.25
Max Drawdown:    -20.0%
Num Trades:      6


In [14]:
for ticker, result in results.items():
    # Plot equity curve + drawdown
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    row_heights=[0.5, 0.25, 0.25],
                    subplot_titles=["Equity Curve", "Drawdown", "Price + MAs"])

    ticker = prices.columns.get_level_values("Ticker").unique()[0]

    fig.add_trace(go.Scatter(x=result.index, y=result["equity"], name="Equity"), row=1, col=1)
    fig.add_trace(go.Scatter(x=result.index, y=result["drawdown"], fill="tozeroy",
                         name="Drawdown", line_color="red"), row=2, col=1)
    fig.add_trace(go.Scatter(x=result.index, y=result["Close"], name="Close", opacity=0.5), row=3, col=1)
    fig.add_trace(go.Scatter(x=result.index, y=result["fast_ma"], name="Fast MA"), row=3, col=1)
    fig.add_trace(go.Scatter(x=result.index, y=result["slow_ma"], name="Slow MA"), row=3, col=1)

    fig.update_layout(height=700, title_text=f"SMA Crossover Backtest — {ticker}")
    fig.show()

In [ ]:
# TODO: play around with different fast/slow MA values. How does the performance change? Number of trades?
# TODO: assume worst case execution: buy at High, sell at Low
# TODO: add more signals: RSI, Bollinger Bands, etc.